# Impact des manifestations sur l'activité de GAB dégradés — version fiche GAB × jour

**Point de départ.** On ne sait pas à l'avance quels GAB ont été touchés : on part d'une **zone** (codes postaux)
ou d'une liste de GAB, et la section 4 trie les GAB : comportement **changé** ou **inchangé** depuis la date des
manifestations. Les GAB changés sont ensuite étudiés en détail.

**Objectif.** Pour chaque GAB étudié, établir son comportement **normal** (habitude par jour de semaine,
saisonnalité, profil horaire, billets, captures), puis démontrer s'il y a eu **rupture** après la dégradation :
GAB hors service, activité réduite, fonctionnement intermittent, ou absence d'impact.

**Entrée** : la fiche d'identité `fiche_gab_jour` (une ligne par GAB et par jour, sur 3 ans). Plus de lecture des
opérations brutes : tout ce dont on a besoin est déjà dans la fiche, l'exécution prend quelques secondes.

**Méthode.**
1. Une ligne par GAB et par jour, calendrier complété par des 0 : c'est l'absence de retraits qui se mesure.
2. **Activité attendue** (contrefactuel) : habitude du GAB par jour de semaine sur les 6 mois précédant la
   dégradation, corrigée de la saisonnalité observée l'année précédente sur la même période.
3. **Date de rupture** : déclarée (si connue) et détectée (premier jour où l'activité sur 7 jours tombe sous
   50 % de l'habitude) ; les deux sont comparées.
4. **KPI avant / après / même période N-1**, test statistique, retraits non servis et montants non distribués.

La détection **règle par règle** (le GAB sort-il dans la liste du jour, à quel rang, avec quelle phrase ?) est
dans le notebook `analyse_descriptive_fiche_gab`, section 7 (`CAS_TESTS`). Ici on mesure l'impact, là on vérifie
qu'on l'aurait vu.

Exécuter les cellules dans l'ordre. Seule la section 1 se modifie.

## 0. Imports

In [ ]:
import math
import warnings
warnings.filterwarnings("ignore")

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dataiku
from IPython.display import HTML, display
from matplotlib.colors import LinearSegmentedColormap

pd.set_option("display.max_columns", 60)

## 1. Configuration
Seul le **bloc à compléter** se modifie. Deux façons de choisir les GAB : une **liste** (`GAB_ETUDIES`) ou une
**zone** (`ZONE_CODES_POSTAUX`, débuts de code postal : `["13001", "13002"]` = deux
arrondissements). Si les dates de dégradation sont inconnues, laisser `{}` : elles sont détectées (section 6).

In [ ]:
# ============================================================================================================
# BLOC À COMPLÉTER
# ============================================================================================================
DATASET_FICHE = "fiche_gab_jour"             # la fiche d'identité GAB x jour
GAB_ETUDIES = None                           # ["num1", "num2"] : liste de GAB ; None = tous les GAB de la zone
ZONE_CODES_POSTAUX = ["00"]                  # débuts de code postal retenus quand GAB_ETUDIES est None (ex. ["75"] = un département)
DATE_MANIFESTATION = None                    # "AAAA-MM-JJ" : date des manifestations (repère et début de recherche)
DATES_DEGRADATION = {}                       # {"num_automate": "AAAA-MM-JJ"} si la date de chaque GAB est connue (fiche de suivi)
NB_GAB_DETAIL_MAX = 12                       # au plus N GAB étudiés en détail (sections 5 à 11) : les changés d'abord

# --- paramètres de la méthode
JOURS_REFERENCE = 182        # période « normale » : 6 mois avant la dégradation...
JOURS_EXCLUS_AVANT = 7       # ... hors la dernière semaine (manifestations déjà en cours)
JOURS_RECENTS = 14           # fenêtre « derniers jours » examinée par le tri (section 4), à partir de DATE_MANIFESTATION
JOURS_RECHERCHE = 60         # sans DATE_MANIFESTATION : rupture recherchée sur les 60 derniers jours
SEUIL_RUPTURE = 0.5          # rupture : activité sur 7 jours < 50 % de l'habitude
SEUIL_REPRISE = 0.8          # reprise : activité sur 7 jours >= 80 % de l'attendu
SEUIL_HORS_SERVICE = 0.1     # hors service : < 10 % de l'attendu sur les 28 derniers jours
DATASET_SORTIE_SYNTHESE = None   # ex. "impact_manifestations_synthese" (dataset à créer dans le Flow)

## 2. Fonctions d'affichage
Tableaux et chiffres clés en HTML simple ; une palette unique : bleu = observé, gris = attendu ou période de
référence, trait noir = dégradation.

In [ ]:
BLEU, GRIS, GRIS_CLAIR, ORANGE, ENCRE = "#2a78d6", "#8a8984", "#d9d8d3", "#eb6834", "#0b0b0b"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#ececea", "grid.linewidth": 0.8, "axes.edgecolor": "#b5b4ae",
                     "axes.titlesize": 11, "axes.titleweight": "bold", "font.size": 9,
                     "axes.prop_cycle": plt.cycler(color=[BLEU, GRIS, ORANGE])})
CSS = """<style>
.kpi-grille{display:flex;flex-wrap:wrap;gap:10px;margin:6px 0 14px}
.kpi{border:1px solid #e3e2de;border-radius:8px;padding:10px 14px;min-width:150px;background:#fcfcfb}
.kpi .v{font-size:22px;font-weight:700;color:#0b0b0b}.kpi .l{font-size:12px;color:#52514e}
table.t{border-collapse:collapse;font-size:12px;margin:4px 0 12px}
table.t th{background:#f3f2ef;color:#0b0b0b;text-align:left;padding:5px 9px;border-bottom:2px solid #d9d8d3}
table.t td{padding:4px 9px;border-bottom:1px solid #ecebe8;text-align:right}
table.t td.txt{text-align:left}.neg{color:#c4380f;font-weight:600}.pos{color:#1a7f37;font-weight:600}
.note{color:#52514e;font-size:12.5px;margin:2px 0 10px;max-width:900px}
</style>"""


def afficher_titre(titre, note=None):
    html = CSS + "<h4 style='margin:10px 0 2px'>{}</h4>".format(titre)
    if note:
        html += "<div class='note'>{}</div>".format(note)
    display(HTML(html))


def formater(v):
    if isinstance(v, (int, np.integer)):
        return "{:,}".format(int(v)).replace(",", " ")
    if isinstance(v, (float, np.floating)):
        if np.isnan(v):
            return "–"
        return "{:,.1f}".format(v).replace(",", " ") if abs(v) < 1000 else "{:,.0f}".format(v).replace(",", " ")
    return "" if v is None else str(v)


def afficher_chiffres(chiffres, titre=None):
    tuiles = "".join("<div class='kpi'><div class='v'>{}</div><div class='l'>{}</div></div>".format(formater(v), k) for k, v in chiffres.items())
    display(HTML(CSS + ("<h4 style='margin:10px 0 2px'>{}</h4>".format(titre) if titre else "") + "<div class='kpi-grille'>{}</div>".format(tuiles)))


def echapper(texte):
    return str(texte).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def afficher_tableau(table, titre=None, note=None, colonnes_variation=(), max_lignes=60):
    """Tableau HTML ; les colonnes de variation (%) sont colorées (baisse en rouge, hausse en vert)."""
    entete = "".join("<th>{}</th>".format(c) for c in table.columns)
    corps = []
    for _, ligne in table.head(max_lignes).iterrows():
        cellules = []
        for c in table.columns:
            v = ligne[c]
            classe = "txt" if isinstance(v, str) else ""
            if c in colonnes_variation and isinstance(v, (int, float, np.number)) and not pd.isna(v):
                classe = "neg" if v <= -20 else ("pos" if v >= 20 else "")
                v = "{:+.0f} %".format(v)
            cellules.append("<td class='{}'>{}</td>".format(classe, echapper(formater(v))))
        corps.append("<tr>{}</tr>".format("".join(cellules)))
    if titre:
        afficher_titre(titre, note)
    display(HTML(CSS + "<table class='t'><tr>{}</tr>{}</table>".format(entete, "".join(corps))))
    if len(table) > max_lignes:
        display(HTML("<div class='note'>… {} lignes au total, {} affichées.</div>".format(len(table), max_lignes)))


def marquer_degradation(ax, date, libelle=True):
    if date is not None and not pd.isna(date):
        ax.axvline(date, color=ENCRE, linewidth=1.2)
        if libelle:
            ax.annotate("dégradation", (date, 1), xycoords=("data", "axes fraction"), xytext=(3, -10), textcoords="offset points", fontsize=8, color=ENCRE)


def axe_dates(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%y"))
    for etiquette in ax.get_xticklabels():
        etiquette.set_fontsize(7)

## 3. Lecture de la fiche et table GAB × jour
Seules les colonnes utiles sont lues. Correspondance avec l'ancienne version (opérations brutes) :
`retraits` = `nb_retraits`, `retraits_annules` = `nb_inaboutis` (retrait tenté non abouti), `captures` = `nb_captures`,
`montant` = `montant_total`, `retraits_confreres` = `nb_retraits − nb_ret_porteur`, `heures_actives` = `nb_heures_actives`,
`operations` = `nb_lignes` (ou retraits + inaboutis + captures si absent). Le profil horaire vient des **tranches**
(0-6 h, 6-9 h, 9-12 h, 12-14 h, 14-18 h, 18-22 h, 22-24 h), les billets des colonnes `nb_billets_*`, les motifs de
capture des colonnes `nb_capture_*`.

In [ ]:
TRANCHES = ["nb_0h_6h", "nb_6h_9h", "nb_9h_12h", "nb_12h_14h", "nb_14h_18h", "nb_18h_22h", "nb_22h_24h"]
LIB_TRANCHES = ["0-6 h", "6-9 h", "9-12 h", "12-14 h", "14-18 h", "18-22 h", "22-24 h"]
BILLETS = ["nb_billets_5", "nb_billets_10", "nb_billets_20", "nb_billets_50", "nb_billets_100"]
MOTIFS = ["nb_capture_oubli_lecture", "nb_capture_capture_cos", "nb_capture_conserver_carte", "nb_capture_carte_perdue", "nb_capture_carte_volee", "nb_capture_autre_motif"]
COLONNES = ["num_automate", "date_jour", "code_site_implantation", "ville", "code_postal", "latitude", "longitude", "statut_gab", "type_gab_e_i",
            "nb_retraits", "nb_lignes", "nb_inaboutis", "nb_captures", "montant_total", "nb_ret_porteur", "nb_heures_actives",
            "heure_premier_retrait", "heure_dernier_retrait", "plus_long_trou_min", "jour_sans_retrait"] + TRANCHES + BILLETS + MOTIFS

_ds = dataiku.Dataset(DATASET_FICHE)
try:
    _presentes = [c["name"].lower() for c in _ds.read_schema()]
    brut = _ds.get_dataframe(columns=[c for c in COLONNES if c in _presentes])
except Exception:
    brut = _ds.get_dataframe()
brut.columns = [c.lower() for c in brut.columns]
brut["num_automate"] = brut["num_automate"].astype(str).str.strip()
brut["date_jour"] = pd.to_datetime(brut["date_jour"].astype(str).str[:10], errors="coerce")
brut = brut[brut["date_jour"].notnull()]
if "code_postal" in brut.columns:
    brut["code_postal"] = brut["code_postal"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)

# choix des GAB : liste, sinon zone
if GAB_ETUDIES:
    brut = brut[brut["num_automate"].isin([str(g) for g in GAB_ETUDIES])]
    CHOIX = "liste de {} GAB".format(len(GAB_ETUDIES))
elif ZONE_CODES_POSTAUX and "code_postal" in brut.columns:
    m_zone = pd.Series(False, index=brut.index)
    for p in ZONE_CODES_POSTAUX:
        m_zone |= brut["code_postal"].str.startswith(str(p).zfill(2) if len(str(p)) <= 2 else str(p))
    gab_zone = brut.loc[m_zone, "num_automate"].unique()
    brut = brut[brut["num_automate"].isin(gab_zone)]
    CHOIX = "zone {} ({} GAB)".format(", ".join(ZONE_CODES_POSTAUX), len(gab_zone))
else:
    CHOIX = "tous les GAB de la fiche"
if brut.empty:
    raise ValueError("Aucun GAB retenu ({}). Vérifier GAB_ETUDIES / ZONE_CODES_POSTAUX.".format(CHOIX))

for c in ["nb_retraits", "nb_lignes", "nb_inaboutis", "nb_captures", "montant_total", "nb_ret_porteur", "nb_heures_actives", "plus_long_trou_min"] + TRANCHES + BILLETS + MOTIFS:
    if c in brut.columns:
        brut[c] = pd.to_numeric(brut[c], errors="coerce").fillna(0)
    else:
        brut[c] = 0.0 if c not in ("nb_lignes",) else np.nan
if brut["nb_lignes"].isnull().all():
    brut["nb_lignes"] = brut["nb_retraits"] + brut["nb_inaboutis"] + brut["nb_captures"]

jour_brut = pd.DataFrame({
    "num_automate": brut["num_automate"], "date": brut["date_jour"],
    "operations": brut["nb_lignes"], "retraits": brut["nb_retraits"], "retraits_annules": brut["nb_inaboutis"],
    "captures": brut["nb_captures"], "montant": brut["montant_total"],
    "retraits_confreres": (brut["nb_retraits"] - brut["nb_ret_porteur"]).clip(lower=0),
    "heures_actives": brut["nb_heures_actives"]})
for c in TRANCHES + BILLETS + MOTIFS:
    jour_brut[c] = brut[c].values
jour_brut = jour_brut.groupby(["num_automate", "date"], as_index=False).sum()      # sécurité : une ligne par GAB et par jour

# fiche d'identité des GAB retenus
_id_cols = [c for c in ["code_site_implantation", "ville", "code_postal", "latitude", "longitude", "statut_gab", "type_gab_e_i"] if c in brut.columns]
fiche = brut.sort_values("date_jour").groupby("num_automate").agg(dict({c: "last" for c in _id_cols},
                                                                       **{"date_jour": ["min", "max"], "nb_retraits": "sum"}))
fiche.columns = _id_cols + ["premiere_journee", "derniere_journee", "retraits_total"]
_actifs = brut[brut["nb_retraits"] > 0].groupby("num_automate")["date_jour"].max().rename("dernier_retrait")
fiche = fiche.join(_actifs).reset_index()
for c in ("premiere_journee", "derniere_journee", "dernier_retrait"):
    fiche[c] = fiche[c].dt.strftime("%d/%m/%Y").fillna("–")
afficher_tableau(fiche, "GAB retenus : {}".format(CHOIX),
                 "Dernier retrait très antérieur à la dernière journée = GAB qui ne fonctionne plus. Si la zone compte des centaines de GAB, "
                 "seules les 60 premières lignes sont affichées ; le tri de la section 4 porte sur tous.", max_lignes=60)

In [ ]:
def completer_calendrier(jour_brut):
    """Calendrier complet pour chaque GAB, de son premier jour au dernier jour des données : les jours absents deviennent des 0."""
    fin = jour_brut["date"].max()
    morceaux = []
    for gab, t in jour_brut.groupby("num_automate"):
        cal = pd.DataFrame({"num_automate": gab, "date": pd.date_range(t["date"].min(), fin, freq="D")})
        morceaux.append(cal.merge(t, on=["num_automate", "date"], how="left"))
    table = pd.concat(morceaux, ignore_index=True)
    comptes = [c for c in table.columns if c not in ("num_automate", "date")]
    table[comptes] = table[comptes].fillna(0)
    table["jour_semaine"] = table["date"].dt.dayofweek
    table["sans_retrait"] = (table["retraits"] == 0).astype(int)
    return table


journees = completer_calendrier(jour_brut)
LISTE_GAB = sorted(journees["num_automate"].unique())
FIN_DONNEES = journees["date"].max()
afficher_chiffres({"GAB retenus": len(LISTE_GAB), "jours couverts": journees["date"].nunique(),
                   "premier jour": journees["date"].min().strftime("%d/%m/%Y"), "dernier jour": FIN_DONNEES.strftime("%d/%m/%Y"),
                   "retraits au total": int(journees["retraits"].sum())}, "Table GAB x jour")

## 4. Tri des GAB : quels GAB ont changé de comportement depuis les manifestations ?
Pour chaque GAB, on compare la **fenêtre examinée** (les `JOURS_RECENTS` jours à partir de `DATE_MANIFESTATION`, ou
les derniers jours des données si elle n'est pas renseignée) à son **habitude** : moyenne des mêmes jours de semaine
des 8 semaines qui précèdent la fenêtre. Trois repères évitent les fausses alertes :
- **rang dans l'année** : le même calcul est refait sur toutes les fenêtres de même durée des 12 derniers mois ;
  un niveau jamais (ou presque jamais) atteint en un an n'est pas une fluctuation ordinaire ;
- **même période l'an dernier** : si l'an dernier la même fenêtre était déjà basse (vacances, Toussaint), la baisse est saisonnière ;
- **autres signaux** : heures d'activité, montant moyen, captures, inaboutis (GAB ouvert mais endommagé).

| Statut | Règle |
|---|---|
| Arrêté | au moins 3 derniers jours de la fenêtre sans retrait, ou moins de 10 % de l'habitude |
| Baisse marquée | sur la fenêtre ou sur ses 7 premiers jours : moins de 70 % de l'habitude, niveau parmi les 2 % les plus bas de l'année, pas saisonnier |
| Hausse marquée | idem à la hausse : plus de 130 %, parmi les 2 % les plus hauts, pas saisonnier (report probable) |
| À surveiller | au moins 3 jours sous 50 % de l'habitude, ou niveau parmi les 2 % extrêmes de l'année avec un écart > 20 %, ou autre signal |
| Inchangé | aucun des cas précédents |

Les GAB **Arrêté**, **Baisse marquée** et **Hausse marquée** (et ceux de `DATES_DEGRADATION`) sont étudiés en détail
dans les sections suivantes, dans la limite de `NB_GAB_DETAIL_MAX`.

In [ ]:
DEBUT_FENETRE = pd.Timestamp(DATE_MANIFESTATION) if DATE_MANIFESTATION else FIN_DONNEES - pd.Timedelta(days=JOURS_RECENTS - 1)
FIN_FENETRE = min(DEBUT_FENETRE + pd.Timedelta(days=JOURS_RECENTS - 1), FIN_DONNEES)
N_FENETRE = (FIN_FENETRE - DEBUT_FENETRE).days + 1


def rapport_fenetre(retraits, jours_semaine, debut, n):
    """Retraits des jours debut..debut+n-1 / habitude (mêmes jours de semaine des 8 semaines avant debut)."""
    if debut < 56 or debut + n > len(retraits):
        return np.nan, np.nan
    reference = pd.Series(retraits[debut - 56:debut]).groupby(jours_semaine[debut - 56:debut]).mean()
    attendu = reference.reindex(jours_semaine[debut:debut + n]).values.sum()
    return (retraits[debut:debut + n].sum() / attendu if attendu > 0 else np.nan), attendu


def indicateurs_periode(t):
    retraits = t["retraits"].sum()
    return {"heures": t["heures_actives"].mean(),
            "montant_moyen": t["montant"].sum() / retraits if retraits else np.nan,
            "captures": 1000 * t["captures"].sum() / max(t["operations"].sum(), 1),
            "annules": 100 * t["retraits_annules"].sum() / max(retraits + t["retraits_annules"].sum(), 1)}


def autres_signaux(recent, reference):
    """Changements de fonctionnement d'un GAB encore actif."""
    r, a = indicateurs_periode(recent), indicateurs_periode(reference)
    signaux = []
    if a["heures"] and r["heures"] < 0.8 * a["heures"]:
        signaux.append("heures d'activité {:.0f} → {:.0f}".format(a["heures"], r["heures"]))
    if a["montant_moyen"] and r["montant_moyen"] < 0.8 * a["montant_moyen"]:
        signaux.append("montant moyen {:.0f} → {:.0f} €".format(a["montant_moyen"], r["montant_moyen"]))
    if r["captures"] >= 3 and r["captures"] > 2 * a["captures"]:
        signaux.append("captures {:.0f} → {:.0f} pour 1000".format(a["captures"], r["captures"]))
    if r["annules"] >= 3 and r["annules"] > 2 * a["annules"]:
        signaux.append("inaboutis {:.1f} → {:.1f} %".format(a["annules"], r["annules"]))
    return signaux


def rapports_glissants(retraits, n):
    """Pour chaque jour J : retraits des n jours J..J+n-1 / habitude (moyenne des mêmes jours de semaine des 8 semaines
    précédant chaque jour). Calcul vectorisé, utilisé pour situer la fenêtre dans l'année (rang)."""
    s = pd.Series(retraits)
    hab = sum(s.shift(7 * k) for k in range(1, 9)) / 8
    obs = s.rolling(n).sum().shift(-(n - 1))
    att = hab.rolling(n).sum().shift(-(n - 1))
    return (obs / att.where(att > 0)).values


def mesurer_fenetre(retraits, jours_semaine, debut, n):
    """Rapport à l'habitude de la fenêtre [debut, debut+n), son rang parmi les fenêtres de n jours des 12 mois
    précédents (sans chevauchement) et le même rapport un an plus tôt."""
    rapport, attendu = rapport_fenetre(retraits, jours_semaine, debut, n)
    historique = rapports_glissants(retraits, n)[max(56, debut - 365):max(56, debut - n + 1)]
    historique = historique[~np.isnan(historique)]
    n_1 = rapport_fenetre(retraits, jours_semaine, debut - 364, n)[0] if debut - 364 >= 56 else np.nan
    return {"rapport": rapport, "attendu": attendu, "n_1": n_1,
            "rang": (historique <= rapport).mean() if len(historique) and not np.isnan(rapport) else np.nan,
            "saisonnier": not np.isnan(n_1) and not np.isnan(rapport) and abs(rapport - n_1) < 0.15}


def trier_gab(journees):
    """Un statut par GAB sur la fenêtre examinée (et sur ses 7 premiers jours), avec les éléments qui le justifient."""
    lignes = []
    for gab, t in journees.groupby("num_automate"):
        t = t.sort_values("date").reset_index(drop=True)
        retraits, jours_semaine = t["retraits"].values.astype(float), t["jour_semaine"].values
        if not (t["date"] == DEBUT_FENETRE).any():
            continue
        debut = int(t.index[t["date"] == DEBUT_FENETRE][0])
        n = min(N_FENETRE, len(t) - debut)
        fenetres = [mesurer_fenetre(retraits, jours_semaine, debut, n), mesurer_fenetre(retraits, jours_semaine, debut, min(7, n))]
        rapport, attendu, n_1 = fenetres[0]["rapport"], fenetres[0]["attendu"], fenetres[0]["n_1"]
        if np.isnan(rapport):
            lignes.append({"GAB": gab, "statut": "Historique insuffisant", "activité / habitude (%)": np.nan, "7 premiers jours (%)": np.nan,
                           "même période N-1 (%)": np.nan, "rang dans l'année (%)": "–", "retraits": int(retraits[debut:debut + n].sum()), "attendus": np.nan,
                           "jours sous 50 %": np.nan, "derniers jours sans retrait": np.nan, "premier jour sous 50 %": "–", "autres signaux": "–"})
            continue
        baisse = any(f["rapport"] < 0.7 and f["rang"] <= 0.02 and not f["saisonnier"] for f in fenetres if not np.isnan(f["rapport"]))
        hausse = any(f["rapport"] > 1.3 and f["rang"] >= 0.98 and not f["saisonnier"] for f in fenetres if not np.isnan(f["rapport"]))
        extreme = any((f["rang"] <= 0.02 or f["rang"] >= 0.98) and (f["rapport"] < 0.8 or f["rapport"] > 1.2) for f in fenetres if not np.isnan(f["rang"]))
        recent, reference = t.iloc[debut:debut + n], t.iloc[debut - 56:debut]
        habitude_jour = reference.groupby("jour_semaine")["retraits"].mean()
        attendu_jour = recent["jour_semaine"].map(habitude_jour)
        jours_bas = int((recent["retraits"] < SEUIL_RUPTURE * attendu_jour).sum())
        zeros_fin = int(len(recent) - recent["retraits"].to_numpy().nonzero()[0].max() - 1) if recent["retraits"].sum() else len(recent)
        premier_bas = recent.loc[recent["retraits"] < SEUIL_RUPTURE * attendu_jour, "date"].min()
        signaux = autres_signaux(recent[recent["retraits"] > 0], reference)
        if zeros_fin >= 3 or rapport < SEUIL_HORS_SERVICE:
            statut = "Arrêté"
        elif baisse:
            statut = "Baisse marquée"
        elif hausse:
            statut = "Hausse marquée"
        elif jours_bas >= 3 or extreme or signaux:
            statut = "À surveiller"
        else:
            statut = "Inchangé"
        pct = lambda v: round(100 * v, 0) if not np.isnan(v) else np.nan
        lignes.append({
            "GAB": gab, "statut": statut, "activité / habitude (%)": pct(rapport), "7 premiers jours (%)": pct(fenetres[1]["rapport"]),
            "même période N-1 (%)": pct(n_1),
            "rang dans l'année (%)": "{} / {}".format("–" if np.isnan(fenetres[0]["rang"]) else "{:.0f}".format(100 * fenetres[0]["rang"]),
                                                      "–" if np.isnan(fenetres[1]["rang"]) else "{:.0f}".format(100 * fenetres[1]["rang"])),
            "retraits": int(recent["retraits"].sum()), "attendus": int(round(attendu)),
            "jours sous 50 %": jours_bas, "derniers jours sans retrait": zeros_fin,
            "premier jour sous 50 %": premier_bas.strftime("%d/%m/%Y") if not pd.isna(premier_bas) else "–",
            "autres signaux": " ; ".join(signaux) or "–"})
    ordre = {"Arrêté": 0, "Baisse marquée": 1, "Hausse marquée": 2, "À surveiller": 3, "Inchangé": 4, "Historique insuffisant": 5}
    table = pd.DataFrame(lignes)
    table["_ordre"] = table["statut"].map(ordre)
    table = table.sort_values(["_ordre", "activité / habitude (%)"]).drop(columns="_ordre").reset_index(drop=True)
    if "ville" in fiche.columns:
        table.insert(1, "ville", table["GAB"].map(fiche.set_index("num_automate")["ville"]))
    return table


tri = trier_gab(journees)
afficher_chiffres({"GAB examinés": len(tri),
                   "comportement changé": int(tri["statut"].isin(["Arrêté", "Baisse marquée", "Hausse marquée"]).sum()),
                   "arrêtés": int((tri["statut"] == "Arrêté").sum()),
                   "à surveiller": int((tri["statut"] == "À surveiller").sum()),
                   "inchangés": int((tri["statut"] == "Inchangé").sum()),
                   "fenêtre examinée": "{} au {}".format(DEBUT_FENETRE.strftime("%d/%m"), FIN_FENETRE.strftime("%d/%m/%Y"))},
                  "Tri des GAB sur la fenêtre de {} jours".format(N_FENETRE))
afficher_tableau(tri, "Statut de chaque GAB (changés en premier)",
                 "activité / habitude : retraits de la fenêtre en % de l'habitude du GAB. Rang dans l'année : part des fenêtres de même durée des 12 "
                 "mois précédents avec une activité inférieure ou égale, sur la fenêtre / sur ses 7 premiers jours (0 % = niveau le plus bas de "
                 "l'année, 100 % = le plus haut). Même période N-1 : le même rapport un an plus tôt ; s'il est proche, la variation est saisonnière.",
                 max_lignes=80)

# GAB étudiés en détail : les changés (puis à surveiller), les déclarés, dans la limite
changes = list(tri.loc[tri["statut"].isin(["Arrêté", "Baisse marquée", "Hausse marquée"]), "GAB"])
declares = [g for g in DATES_DEGRADATION if g in LISTE_GAB]
surveiller = list(tri.loc[tri["statut"] == "À surveiller", "GAB"])
GAB_DETAIL = []
for g in declares + changes + surveiller + (list(GAB_ETUDIES) if GAB_ETUDIES else []):
    if g in LISTE_GAB and g not in GAB_DETAIL and len(GAB_DETAIL) < NB_GAB_DETAIL_MAX:
        GAB_DETAIL.append(g)
if not GAB_DETAIL:
    GAB_DETAIL = list(tri["GAB"].head(NB_GAB_DETAIL_MAX))
display(HTML(CSS + "<div class='note'><b>{} GAB étudiés en détail</b> dans les sections suivantes : {}</div>".format(len(GAB_DETAIL), ", ".join(GAB_DETAIL))))

In [ ]:
def tracer_derniers_jours(journees, tri, gabs, jours=90):
    """Un panneau par GAB : retraits du jour et habitude, fenêtre examinée en fond."""
    statuts = tri.set_index("GAB")["statut"]
    fig, axes = plt.subplots(len(gabs), 1, figsize=(13, 2.1 * len(gabs)), sharex=True)
    for ax, gab in zip(np.atleast_1d(axes), gabs):
        t = journees[journees["num_automate"] == gab].set_index("date")["retraits"]
        habitude = sum(t.shift(7 * k) for k in range(1, 9)) / 8
        habitude[habitude.index >= DEBUT_FENETRE] = np.nan     # figée : habitude de la fenêtre = 8 semaines avant
        gel = t[(t.index >= DEBUT_FENETRE - pd.Timedelta(days=56)) & (t.index < DEBUT_FENETRE)]
        moyennes = gel.groupby(gel.index.dayofweek).mean()
        fenetre = t.index[t.index >= DEBUT_FENETRE]
        habitude.loc[fenetre] = moyennes.reindex(fenetre.dayofweek).values
        deb = max(DEBUT_FENETRE - pd.Timedelta(days=jours - N_FENETRE), t.index.min())
        t, habitude = t[t.index >= deb], habitude[habitude.index >= deb]
        ax.axvspan(DEBUT_FENETRE - pd.Timedelta(hours=12), FIN_FENETRE + pd.Timedelta(hours=12), color="#f3f2ef", zorder=0)
        change = statuts.get(gab, "") in ("Arrêté", "Baisse marquée", "Hausse marquée")
        ax.bar(t.index, t.values, width=0.8, color=BLEU if change else GRIS_CLAIR, linewidth=0, zorder=2, label="retraits du jour")
        ax.plot(habitude.index, habitude.values, color=ENCRE if change else GRIS, linewidth=1.4, linestyle="--", zorder=3, label="habitude")
        if DATE_MANIFESTATION:
            marquer_degradation(ax, pd.Timestamp(DATE_MANIFESTATION))
        ax.set_title("{}  –  {}".format(gab, statuts.get(gab, "")), loc="left")
        ax.set_ylim(bottom=0)
        ax.set_ylabel("retraits / jour")
    ax = np.atleast_1d(axes)[-1]
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
    np.atleast_1d(axes)[0].legend(loc="upper left", ncol=2, frameon=False, fontsize=8, bbox_to_anchor=(0.3, 1.3))
    fig.tight_layout()
    plt.show()


afficher_titre("Autour de la fenêtre examinée, GAB par GAB",
               "Fond gris : fenêtre examinée. Pointillés : habitude du GAB pour chaque jour. En bleu, les GAB dont le comportement a changé ; en gris clair, les autres.")
tracer_derniers_jours(journees, tri, GAB_DETAIL)

In [ ]:
def tracer_calendrier_recent(journees, gabs, jours_avant=21):
    """GAB x jours : retraits en % de l'habitude (mêmes jours de semaine des 8 semaines précédentes), 3 semaines avant la fenêtre et jusqu'à la fin."""
    debut = DEBUT_FENETRE - pd.Timedelta(days=jours_avant)
    lignes = {}
    for gab in gabs:
        t = journees[journees["num_automate"] == gab].set_index("date")["retraits"]
        habitude = sum(t.shift(7 * k) for k in range(1, 9)) / 8
        lignes[gab] = (100 * t / habitude.where(habitude > 0))[t.index >= debut]
    pivot = pd.DataFrame(lignes).T
    pivot = pivot[[c for c in pivot.columns if (c - debut).days <= jours_avant + 56]]        # au plus 11 semaines affichées
    palette = LinearSegmentedColormap.from_list("ecart", [ORANGE, "#f2f1ee", BLEU])
    fig, ax = plt.subplots(figsize=(13, 0.45 * len(pivot) + 1.3))
    image = ax.imshow(pivot.clip(0, 200).values, aspect="auto", cmap=palette, vmin=0, vmax=200, interpolation="nearest")
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index)
    ax.set_xticks(range(len(pivot.columns)))
    jl = ["lu", "ma", "me", "je", "ve", "sa", "di"]
    ax.set_xticklabels(["{} {}".format(jl[d.dayofweek], d.strftime("%d/%m")) for d in pivot.columns], rotation=90, fontsize=7)
    if DEBUT_FENETRE in pivot.columns:
        ax.axvline(list(pivot.columns).index(DEBUT_FENETRE) - 0.5, color=ENCRE, linewidth=1)
    ax.grid(False)
    barre = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.01)
    barre.set_label("% de l'habitude")
    fig.tight_layout()
    plt.show()


afficher_titre("Jour par jour, en % de l'habitude",
               "Gris clair = journée normale (100 %), orange = sous l'habitude, orange vif = aucun retrait, bleu = au-dessus. Le trait noir marque "
               "le début de la fenêtre (manifestations). Un GAB touché apparaît comme une bande orange qui commence le même jour pour plusieurs GAB.")
tracer_calendrier_recent(journees, GAB_DETAIL)

## 5. Comportement normal des GAB étudiés
Activité hebdomadaire sur toute la période, profil par jour de semaine et saisonnalité mensuelle : c'est la
référence à laquelle la période après dégradation est comparée.

In [ ]:
LISTE_GAB = GAB_DETAIL        # à partir d'ici, seuls les GAB étudiés en détail


def tracer_historique(journees, dates):
    n = len(LISTE_GAB)
    fig, axes = plt.subplots(n, 1, figsize=(13, 1.9 * n), sharex=True)
    for ax, gab in zip(np.atleast_1d(axes), LISTE_GAB):
        jours = journees[journees["num_automate"] == gab].set_index("date")["retraits"].resample("W")
        semaine = jours.sum()[jours.count() == 7]
        ax.fill_between(semaine.index, semaine.values, color=BLEU, alpha=0.15, linewidth=0)
        ax.plot(semaine.index, semaine.values, color=BLEU, linewidth=1.6)
        marquer_degradation(ax, dates.get(gab), libelle=False)
        ax.set_title(gab, loc="left")
        ax.set_ylabel("retraits / sem.")
        ax.set_ylim(bottom=0)
    np.atleast_1d(axes)[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    np.atleast_1d(axes)[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%y"))
    fig.tight_layout()
    plt.show()


afficher_titre("Retraits par semaine depuis le début des données",
               "Chaque panneau a sa propre échelle verticale. Le trait noir marque la date de dégradation déclarée, ou à défaut la date des "
               "manifestations. Les creux de fin d'année ou d'été qui se répètent chaque année relèvent de la saisonnalité, pas d'un incident.")
tracer_historique(journees, {gab: pd.Timestamp(DATES_DEGRADATION.get(gab, DATE_MANIFESTATION)) for gab in LISTE_GAB if DATES_DEGRADATION.get(gab, DATE_MANIFESTATION)})

In [ ]:
def profils_normaux(journees, fin_reference):
    lignes = []
    for gab in LISTE_GAB:
        t = journees[(journees["num_automate"] == gab) & (journees["date"] < fin_reference.get(gab, FIN_DONNEES))]
        semaine = t.groupby("jour_semaine")["retraits"].mean()
        lignes.append(pd.Series(semaine.values / semaine.mean() * 100, index=["lun", "mar", "mer", "jeu", "ven", "sam", "dim"][:len(semaine)], name=gab))
    return pd.DataFrame(lignes).round(0)


def tracer_saisonnalite(journees):
    n = len(LISTE_GAB)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 2.6), sharey=False)
    teintes = {2024: GRIS_CLAIR, 2025: GRIS, 2026: BLEU}
    for ax, gab in zip(np.atleast_1d(axes), LISTE_GAB):
        t = journees[journees["num_automate"] == gab]
        mensuel = t.groupby([t["date"].dt.year, t["date"].dt.month])["retraits"].mean().unstack(0)
        for annee in mensuel.columns:
            ax.plot(mensuel.index, mensuel[annee], color=teintes.get(annee, ORANGE), linewidth=2, marker="o", markersize=3, label=str(annee))
        ax.set_title(gab, loc="left")
        ax.set_xticks(range(1, 13))
        ax.set_xticklabels(list("JFMAMJJASOND"), fontsize=7)
        ax.set_ylim(bottom=0)
    np.atleast_1d(axes)[0].set_ylabel("retraits / jour")
    np.atleast_1d(axes)[-1].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    plt.show()


afficher_titre("Saisonnalité : retraits moyens par jour, mois par mois",
               "Une courbe par année. Les années précédentes se superposent quand le comportement est stable ; l'écart de l'année en cours après la dégradation est l'impact recherché.")
tracer_saisonnalite(journees)
fin_ref_declaree = {gab: pd.Timestamp(d) for gab, d in DATES_DEGRADATION.items()}
afficher_tableau(profils_normaux(journees, fin_ref_declaree).reset_index().rename(columns={"index": "GAB"}),
                 "Profil par jour de semaine (indice 100 = moyenne du GAB)",
                 "Habitude de chaque GAB avant la dégradation : un indice de 130 le samedi signifie 30 % de retraits de plus qu'un jour moyen.")

## 6. Date de rupture : déclarée et détectée
**Détection.** Pour chaque jour J, on compare les retraits des 7 jours J à J+6 à l'habitude : moyenne des mêmes jours
de semaine des 8 semaines précédant J. La rupture est le premier jour où ce rapport tombe sous `SEUIL_RUPTURE` (50 %),
puis on retient, dans cette semaine, le premier jour lui-même sous 50 % de son habitude. Si `DATE_MANIFESTATION` est
renseignée, la recherche se limite à 15 jours avant et 60 jours après ; sinon aux `JOURS_RECHERCHE` derniers jours.

**Date retenue** : la date déclarée si elle existe, sinon la date détectée, sinon la date des manifestations.

In [ ]:
def rapport_habitude(serie):
    observes = [serie.shift(-d) for d in range(7)]
    habitudes = [(sum(serie.shift(7 * k - d) for k in range(1, 9)) / 8).where(observes[d].notna()) for d in range(7)]
    observe_7j = sum(o.fillna(0) for o in observes)
    habitude_7j = sum(h.fillna(0) for h in habitudes)
    disponibles = sum(o.notna().astype(int) for o in observes)
    return (observe_7j / habitude_7j.where(habitude_7j > 0)).where(disponibles >= 3)


def detecter_ruptures(journees, date_manif=None):
    lignes = []
    for gab in LISTE_GAB:
        serie = journees[journees["num_automate"] == gab].set_index("date")["retraits"]
        rapport = rapport_habitude(serie)
        if date_manif:
            debut, fin = pd.Timestamp(date_manif) - pd.Timedelta(days=15), pd.Timestamp(date_manif) + pd.Timedelta(days=60)
        else:
            debut, fin = FIN_DONNEES - pd.Timedelta(days=JOURS_RECHERCHE - 1), FIN_DONNEES
        candidats = rapport[(rapport.index >= debut) & (rapport.index <= fin) & (rapport < SEUIL_RUPTURE)]
        date = candidats.index[0] if len(candidats) else pd.NaT
        if len(candidats):
            habitude_jour = sum(serie.shift(7 * k) for k in range(1, 9)) / 8
            semaine = serie[date:date + pd.Timedelta(days=6)]
            sous_seuil = semaine[semaine < SEUIL_RUPTURE * habitude_jour.reindex(semaine.index)]
            date = sous_seuil.index[0] if len(sous_seuil) else date
        lignes.append({"num_automate": gab, "rupture_detectee": date, "rapport_7j_pct": round(100 * candidats.iloc[0], 0) if len(candidats) else np.nan})
    return pd.DataFrame(lignes)


ruptures = detecter_ruptures(journees, DATE_MANIFESTATION)
ruptures["date_declaree"] = ruptures["num_automate"].map(lambda g: pd.Timestamp(DATES_DEGRADATION[g]) if g in DATES_DEGRADATION else pd.NaT)
secours = pd.Timestamp(DATE_MANIFESTATION) if DATE_MANIFESTATION else ruptures["rupture_detectee"].min()
ruptures["date_retenue"] = ruptures["date_declaree"].fillna(ruptures["rupture_detectee"]).fillna(secours)
ruptures["origine"] = np.where(ruptures["date_declaree"].notna(), "déclarée", np.where(ruptures["rupture_detectee"].notna(), "détectée", "aucune rupture : date des manifestations"))
ruptures["ecart_jours"] = (ruptures["rupture_detectee"] - ruptures["date_declaree"]).dt.days
DATES = dict(zip(ruptures["num_automate"], ruptures["date_retenue"]))
affichage = ruptures.copy()
for c in ("rupture_detectee", "date_declaree", "date_retenue"):
    affichage[c] = affichage[c].dt.strftime("%d/%m/%Y").fillna("–")
afficher_tableau(affichage[["num_automate", "date_declaree", "rupture_detectee", "rapport_7j_pct", "ecart_jours", "date_retenue", "origine"]],
                 "Date de rupture par GAB",
                 "rapport_7j_pct = activité des 7 jours suivant la rupture en % de l'habitude. Un écart de 0 à quelques jours entre date déclarée "
                 "et date détectée confirme que la baisse coïncide avec la dégradation.")

## 7. Activité attendue et activité observée
**Attendu (contrefactuel)** : habitude du GAB par jour de semaine sur la période de référence (6 mois avant la date
retenue, hors dernière semaine), multipliée par un indice de saisonnalité : rapport, l'année précédente, entre
l'activité autour du même jour (28 jours centrés) et l'activité moyenne sur la période de référence décalée d'un an.

In [ ]:
def activite_attendue(journees, dates):
    morceaux = []
    for gab in LISTE_GAB:
        t = journees[journees["num_automate"] == gab].set_index("date").copy()
        date = dates[gab]
        debut_ref = date - pd.Timedelta(days=JOURS_REFERENCE + JOURS_EXCLUS_AVANT)
        fin_ref = date - pd.Timedelta(days=JOURS_EXCLUS_AVANT)
        reference = t[(t.index >= debut_ref) & (t.index < fin_ref)]
        habitude = reference.groupby("jour_semaine")["retraits"].mean()
        lisse = t["retraits"].rolling(28, center=True, min_periods=14).mean()
        n_1 = lisse.shift(364)
        base_n_1 = t["retraits"].shift(364)[(t.index >= debut_ref) & (t.index < fin_ref)].mean()
        indice = (n_1 / base_n_1) if base_n_1 and base_n_1 > 0 else pd.Series(1.0, index=t.index)
        t["indice_saison"] = indice.fillna(1.0).clip(0.5, 2.0)
        t["attendu"] = t["jour_semaine"].map(habitude).values * t["indice_saison"]
        t["montant_attendu"] = t["attendu"] * reference["montant"].sum() / max(reference["retraits"].sum(), 1)
        t["periode"] = np.where(t.index >= date, "après", np.where((t.index >= debut_ref) & (t.index < fin_ref), "référence", "avant"))
        t["jours_depuis"] = (t.index - date).days
        morceaux.append(t.reset_index())
    table = pd.concat(morceaux, ignore_index=True)
    table["attendu_7j"] = table.groupby("num_automate")["attendu"].transform(lambda s: s.rolling(7, 1).mean())
    table["observe_7j"] = table.groupby("num_automate")["retraits"].transform(lambda s: s.rolling(7, 1).mean())
    table["rapport_7j"] = table["observe_7j"] / table["attendu_7j"].where(table["attendu_7j"] > 0)
    return table


journees_detail = activite_attendue(journees[journees["num_automate"].isin(LISTE_GAB)], DATES)
controle = journees_detail[journees_detail["periode"] == "référence"].groupby("num_automate").apply(
    lambda t: pd.Series({"retraits observés": t["retraits"].sum(), "retraits attendus": round(t["attendu"].sum()),
                         "écart %": round(100 * (t["retraits"].sum() / max(t["attendu"].sum(), 1e-9) - 1), 1)})).reset_index()
afficher_tableau(controle, "Contrôle du contrefactuel sur la période de référence",
                 "Sur la période normale, observé et attendu doivent coïncider (écart proche de 0 %) : c'est la preuve que l'attendu reproduit bien l'habitude du GAB.")

In [ ]:
def tracer_observe_attendu(journees, jours_avant=120):
    n = len(LISTE_GAB)
    fig, axes = plt.subplots(n, 1, figsize=(13, 2.3 * n))
    for ax, gab in zip(np.atleast_1d(axes), LISTE_GAB):
        t = journees[(journees["num_automate"] == gab) & (journees["jours_depuis"] >= -jours_avant)]
        ax.bar(t["date"], t["retraits"], width=0.8, color=GRIS_CLAIR, linewidth=0, label="retraits du jour")
        ax.plot(t["date"], t["attendu_7j"], color=GRIS, linewidth=1.6, linestyle="--", label="attendu (7 j)")
        ax.plot(t["date"], t["observe_7j"], color=BLEU, linewidth=2, label="observé (7 j)")
        marquer_degradation(ax, DATES[gab])
        ax.set_title(gab, loc="left")
        ax.set_ylim(bottom=0)
        ax.set_ylabel("retraits / jour")
        axe_dates(ax)
    np.atleast_1d(axes)[0].legend(loc="upper left", ncol=3, frameon=False, fontsize=8, bbox_to_anchor=(0.12, 1.25))
    fig.tight_layout()
    plt.show()


afficher_titre("Observé et attendu autour de la dégradation",
               "Barres grises : retraits du jour. Pointillés : activité attendue sans incident (habitude + saisonnalité). Courbe bleue : activité observée. "
               "L'écart entre les deux après le trait noir est l'impact.")
tracer_observe_attendu(journees_detail)

In [ ]:
def tracer_carte_chaleur(journees, avant=60, apres=120):
    pivot = (journees[(journees["jours_depuis"] >= -avant) & (journees["jours_depuis"] <= apres)]
             .assign(pct=lambda t: 100 * t["retraits"] / t["attendu"].where(t["attendu"] > 0))
             .pivot_table(index="num_automate", columns="jours_depuis", values="pct"))
    palette = LinearSegmentedColormap.from_list("ecart", [ORANGE, "#f2f1ee", BLEU])
    fig, ax = plt.subplots(figsize=(13, 0.45 * len(pivot) + 1.2))
    image = ax.imshow(pivot.clip(0, 200).values, aspect="auto", cmap=palette, vmin=0, vmax=200, interpolation="nearest")
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index)
    pas = list(range(0, pivot.shape[1], 15))
    ax.set_xticks(pas)
    ax.set_xticklabels([pivot.columns[i] for i in pas])
    if 0 in list(pivot.columns):
        ax.axvline(list(pivot.columns).index(0) - 0.5, color=ENCRE, linewidth=1.2)
    ax.set_xlabel("jours depuis la dégradation (0 = date retenue)")
    ax.grid(False)
    barre = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.01)
    barre.set_label("% de l'attendu")
    fig.tight_layout()
    plt.show()


afficher_titre("Activité quotidienne en % de l'attendu, alignée sur la date de dégradation",
               "Chaque ligne est un GAB, chaque colonne un jour. Gris clair = normal (100 %), orange = sous l'attendu, orange vif = aucun retrait. "
               "Une bande orange continue après le trait = GAB hors service ; des rayures = fonctionnement intermittent.")
tracer_carte_chaleur(journees_detail)

## 8. KPI avant / après / même période N-1
Période **avant** = période de référence ; **après** = de la date retenue à la fin des données ; **N-1** = mêmes
dates que la période après, un an plus tôt (contrôle de la saisonnalité).

In [ ]:
def kpi(t):
    n = max(len(t), 1)
    retraits = t["retraits"].sum()
    return {"retraits / jour": retraits / n,
            "montant / jour (€)": t["montant"].sum() / n,
            "montant moyen (€)": t["montant"].sum() / retraits if retraits else np.nan,
            "jours sans retrait (%)": 100 * t["sans_retrait"].mean() if len(t) else np.nan,
            "heures actives / jour": t["heures_actives"].mean(),
            "retraits inaboutis (%)": 100 * t["retraits_annules"].sum() / max(retraits + t["retraits_annules"].sum(), 1),
            "captures / 1000 opérations": 1000 * t["captures"].sum() / max(t["operations"].sum(), 1),
            "part de confrères (%)": 100 * t["retraits_confreres"].sum() / retraits if retraits else np.nan}


def tableau_kpi(journees):
    lignes = []
    for gab in LISTE_GAB:
        t = journees[journees["num_automate"] == gab]
        apres = t[t["periode"] == "après"]
        n_1 = t[t["date"].isin(apres["date"] - pd.Timedelta(days=364))]
        k_avant, k_apres, k_n1 = kpi(t[t["periode"] == "référence"]), kpi(apres), kpi(n_1) if len(n_1) else {}
        for nom in k_avant:
            avant, ap, n1 = k_avant[nom], k_apres[nom], k_n1.get(nom, np.nan)
            variation = lambda a, b: 100 * (a / b - 1) if b and not pd.isna(b) and not pd.isna(a) else np.nan
            lignes.append({"GAB": gab, "KPI": nom, "avant": avant, "après": ap, "N-1 même période": n1, "après vs avant": variation(ap, avant), "après vs N-1": variation(ap, n1)})
    return pd.DataFrame(lignes)


kpis = tableau_kpi(journees_detail)
for gab in LISTE_GAB:
    afficher_tableau(kpis[kpis["GAB"] == gab].drop(columns="GAB"), "KPI {} (dégradation retenue le {})".format(gab, DATES[gab].strftime("%d/%m/%Y")),
                     colonnes_variation=("après vs avant", "après vs N-1"))

In [ ]:
def tracer_kpi_principaux(kpis):
    noms = ["retraits / jour", "jours sans retrait (%)", "heures actives / jour", "captures / 1000 opérations"]
    fig, axes = plt.subplots(1, len(noms), figsize=(15, 3.2))
    x = np.arange(len(LISTE_GAB))
    for ax, nom in zip(axes, noms):
        t = kpis[kpis["KPI"] == nom].set_index("GAB").reindex(LISTE_GAB)
        ax.bar(x - 0.2, t["avant"], width=0.38, color=GRIS, label="avant", edgecolor="white", linewidth=1)
        ax.bar(x + 0.2, t["après"], width=0.38, color=BLEU, label="après", edgecolor="white", linewidth=1)
        ax.set_title(nom, loc="left")
        ax.set_xticks(x)
        ax.set_xticklabels([g[-4:] for g in LISTE_GAB], fontsize=8)
        ax.xaxis.grid(False)
    axes[0].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    plt.show()


afficher_titre("Avant / après : les 4 KPI principaux", "Gris = période de référence, bleu = depuis la dégradation. GAB repérés par les 4 derniers caractères.")
tracer_kpi_principaux(kpis)

## 9. Changement de comportement dans la journée et dans les opérations
Un GAB peut rester ouvert mais fonctionner différemment : plages horaires réduites, une valeur de billet qui
disparaît (cassette vide ou hors service), plus de captures ou d'inaboutis (lecteur ou distributeur endommagé).

In [ ]:
def tracer_profils_horaires(journees):
    """Retraits moyens par tranche horaire, avant (référence) et après, un panneau par GAB."""
    n = len(LISTE_GAB)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 2.6), sharey=False)
    for ax, gab in zip(np.atleast_1d(axes), LISTE_GAB):
        for periode, couleur in (("référence", GRIS), ("après", BLEU)):
            t = journees[(journees["num_automate"] == gab) & (journees["periode"] == periode)]
            if len(t):
                s = t[TRANCHES].sum() / len(t)
                ax.plot(range(len(TRANCHES)), s.values, color=couleur, linewidth=2, marker="o", markersize=3, label="avant" if periode == "référence" else "après")
        ax.set_title(gab, loc="left")
        ax.set_xticks(range(len(TRANCHES)))
        ax.set_xticklabels(LIB_TRANCHES, fontsize=6, rotation=45)
        ax.set_ylim(bottom=0)
    np.atleast_1d(axes)[0].set_ylabel("retraits / jour dans la tranche")
    np.atleast_1d(axes)[-1].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    plt.show()


afficher_titre("Profil horaire avant / après (par tranche)",
               "Retraits moyens par jour dans chaque tranche horaire. Une courbe « après » plate = GAB arrêté ; une courbe abaissée de la même forme = "
               "baisse de fréquentation ; une forme différente = plages horaires modifiées.")
tracer_profils_horaires(journees_detail)

In [ ]:
def tableau_billets(journees, dates):
    """Part des billets délivrés par valeur, avant et après (en %)."""
    lignes = []
    for gab in LISTE_GAB:
        t = journees[journees["num_automate"] == gab]
        for periode in ("référence", "après"):
            s = t.loc[t["periode"] == periode, BILLETS].sum()
            if s.sum():
                ligne = {"GAB": gab, "période": "avant" if periode == "référence" else "après"}
                ligne.update({c.replace("nb_billets_", "") + " €": round(100 * v / s.sum(), 1) for c, v in s.items()})
                lignes.append(ligne)
    return pd.DataFrame(lignes).fillna(0)


afficher_tableau(tableau_billets(journees_detail, DATES), "Billets délivrés par valeur (% des billets)",
                 "Une valeur qui passe à 0 % après la dégradation = cassette vide ou hors service : le GAB fonctionne en mode dégradé (montants limités).")


def tableau_motifs(journees, dates):
    """Captures par motif, par mois en moyenne, avant et après."""
    lignes = []
    for gab in LISTE_GAB:
        t = journees[journees["num_automate"] == gab]
        avant, apres = t[t["periode"] == "référence"], t[t["periode"] == "après"]
        n_avant, n_apres = max(len(avant) / 30.4, 1e-9), max(len(apres) / 30.4, 1e-9)
        for motif in MOTIFS:
            if avant[motif].sum() + apres[motif].sum() > 0:
                lignes.append({"GAB": gab, "motif": motif.replace("nb_capture_", ""), "captures / mois avant": avant[motif].sum() / n_avant,
                               "captures / mois après": apres[motif].sum() / n_apres})
    table = pd.DataFrame(lignes)
    if len(table):
        table["variation"] = 100 * (table["captures / mois après"] / table["captures / mois avant"].where(table["captures / mois avant"] > 0) - 1)
    return table


afficher_tableau(tableau_motifs(journees_detail, DATES), "Captures de carte par motif, avant / après",
                 "Une hausse des « oubli lecture » ou « conserver carte » après la dégradation signale un lecteur de carte endommagé.", colonnes_variation=("variation",))

## 10. Preuve statistique, durée d'indisponibilité et manque à gagner
**Test de Mann-Whitney** : l'activité quotidienne rapportée à l'attendu (corrigée du jour de semaine et de la saison)
est-elle significativement plus basse après la dégradation que pendant la référence ? (test sur les rangs ; p < 0,01
= baisse significative). **Retraits non servis** = attendus − observés, cumulés depuis la dégradation ; **montant non
distribué** = même calcul sur les montants.

In [ ]:
def mann_whitney(avant, apres):
    avant, apres = np.asarray(avant, float), np.asarray(apres, float)
    avant, apres = avant[~np.isnan(avant)], apres[~np.isnan(apres)]
    if len(avant) < 5 or len(apres) < 5:
        return np.nan, np.nan
    rangs = pd.Series(np.concatenate([avant, apres])).rank().values
    n1, n2 = len(avant), len(apres)
    u = rangs[n1:].sum() - n2 * (n2 + 1) / 2
    ecart = math.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    z = (u - n1 * n2 / 2) / ecart
    return z, 0.5 * math.erfc(-z / math.sqrt(2))


def date_reprise(t):
    apres = t[t["periode"] == "après"].reset_index(drop=True)
    for i in range(len(apres)):
        fenetre = apres["rapport_7j"].iloc[i:i + 14]
        if apres["rapport_7j"].iloc[i] >= SEUIL_REPRISE and len(fenetre) == 14 and (fenetre >= SEUIL_RUPTURE).all():
            return apres["date"].iloc[i] - pd.Timedelta(days=6)
    return pd.NaT


def diagnostic(journees, ruptures):
    lignes = []
    for gab in LISTE_GAB:
        t = journees[journees["num_automate"] == gab]
        ref, apres = t[t["periode"] == "référence"], t[t["periode"] == "après"]
        recent = apres.tail(28)
        rapport_apres = apres["retraits"].sum() / max(apres["attendu"].sum(), 1e-9)
        rapport_recent = recent["retraits"].sum() / max(recent["attendu"].sum(), 1e-9)
        a_rupture = not pd.isna(ruptures.set_index("num_automate").loc[gab, "rupture_detectee"])
        reprise = date_reprise(t) if a_rupture else pd.NaT
        part_zero = apres["sans_retrait"].mean() if len(apres) else 0
        if rapport_recent < SEUIL_HORS_SERVICE:
            statut = "Hors service"
        elif not a_rupture and rapport_apres >= 1.2:
            statut = "Hausse d'activité (report probable)"
        elif not a_rupture and rapport_apres >= SEUIL_REPRISE:
            statut = "Pas d'impact visible"
        elif not pd.isna(reprise) and rapport_recent >= SEUIL_REPRISE:
            statut = "Remis en service"
        elif part_zero >= 0.3:
            statut = "Fonctionnement intermittent"
        else:
            statut = "Activité réduite"
        indisponible = apres["retraits"] < SEUIL_HORS_SERVICE * apres["attendu"]
        manque = max(apres["attendu"].sum() - apres["retraits"].sum(), 0)
        manque_montant = max(apres["montant_attendu"].sum() - apres["montant"].sum(), 0)
        z, p = mann_whitney(ref["retraits"] / ref["attendu"].where(ref["attendu"] > 0), apres["retraits"] / apres["attendu"].where(apres["attendu"] > 0))
        derniere = t.loc[t["retraits"] > 0, "date"].max()
        lignes.append({
            "GAB": gab, "statut": statut, "date retenue": DATES[gab].strftime("%d/%m/%Y"),
            "date de reprise": reprise.strftime("%d/%m/%Y") if not pd.isna(reprise) else "–",
            "jours d'indisponibilité": int(indisponible.sum()),
            "dernier retrait": derniere.strftime("%d/%m/%Y") if not pd.isna(derniere) else "–",
            "activité depuis (% attendu)": round(100 * rapport_apres, 0),
            "activité 28 derniers jours (%)": round(100 * rapport_recent, 0),
            "retraits non servis": int(round(manque)), "montant non distribué (€)": int(round(manque_montant)),
            "baisse significative": "–" if np.isnan(p) else ("oui (p < 0,001)" if p < 0.001 else ("oui (p = {:.3f})".format(p) if p < 0.01 else "non (p = {:.2f})".format(p)))})
    return pd.DataFrame(lignes)


synthese = diagnostic(journees_detail, ruptures)
afficher_tableau(synthese, "Diagnostic par GAB",
                 "Jours d'indisponibilité : jours depuis la dégradation avec moins de 10 % de l'activité attendue. Baisse significative : test de "
                 "Mann-Whitney, p-valeur inférieure à 0,01. Retraits non servis et montant non distribué : attendu moins observé, cumulés depuis la dégradation (estimation).")

In [ ]:
def tracer_manque(synthese, journees):
    fig, ax = plt.subplots(figsize=(11, 3.6))
    for gab in synthese.sort_values("retraits non servis", ascending=False)["GAB"]:
        t = journees[(journees["num_automate"] == gab) & (journees["periode"] == "après")]
        if t.empty:
            continue
        cumul = (t["attendu"] - t["retraits"]).cumsum()
        impact = synthese.set_index("GAB").loc[gab, "statut"] not in ("Pas d'impact visible", "Hausse d'activité (report probable)")
        ax.plot(t["jours_depuis"], cumul, color=BLEU if impact else GRIS, linewidth=2 if impact else 1.2)
        ax.annotate(gab, (t["jours_depuis"].iloc[-1], cumul.iloc[-1]), xytext=(4, 0), textcoords="offset points", fontsize=8, va="center", color=ENCRE)
    ax.set_xlabel("jours depuis la dégradation")
    ax.set_ylabel("retraits non servis (cumul)")
    ax.axhline(0, color=GRIS, linewidth=0.8)
    fig.tight_layout()
    plt.show()


afficher_titre("Retraits non servis cumulés depuis la dégradation",
               "Une courbe qui monte en ligne droite = GAB totalement arrêté ; une courbe qui se stabilise = GAB remis en service ; une courbe qui "
               "descend sous 0 = activité au-dessus de l'attendu. En gris : GAB sans impact (ou en hausse).")
tracer_manque(synthese, journees_detail)

## 11. Synthèse
Chiffres clés des GAB étudiés et une phrase par GAB, reprenables telles quelles dans une présentation.

In [ ]:
def phrase(ligne):
    ville = fiche.set_index("num_automate")["ville"].get(ligne["GAB"], "") if "ville" in fiche.columns else ""
    debut = "{} ({}) : ".format(ligne["GAB"], ville or "")
    if ligne["statut"] == "Hors service":
        texte = "hors service depuis le {} (dernier retrait le {}), {} jours d'indisponibilité".format(ligne["date retenue"], ligne["dernier retrait"], ligne["jours d'indisponibilité"])
    elif ligne["statut"] == "Remis en service":
        texte = "arrêté le {}, remis en service le {} après {} jours d'indisponibilité".format(ligne["date retenue"], ligne["date de reprise"], ligne["jours d'indisponibilité"])
    elif ligne["statut"] == "Fonctionnement intermittent":
        texte = "fonctionnement intermittent depuis le {} ({} jours d'indisponibilité), activité à {:.0f} % de l'attendu".format(
            ligne["date retenue"], ligne["jours d'indisponibilité"], ligne["activité depuis (% attendu)"])
    elif ligne["statut"] == "Activité réduite":
        texte = "activité réduite à {:.0f} % de l'attendu depuis le {}".format(ligne["activité depuis (% attendu)"], ligne["date retenue"])
    elif ligne["statut"].startswith("Hausse"):
        return debut + "activité en hausse, à {:.0f} % de l'attendu depuis le {} : report probable des clients des GAB touchés.".format(ligne["activité depuis (% attendu)"], ligne["date retenue"])
    else:
        return debut + "pas d'impact visible : activité à {:.0f} % de l'attendu.".format(ligne["activité depuis (% attendu)"])
    return debut + texte + " ; {} retraits non servis, environ {} € non distribués.".format(formater(ligne["retraits non servis"]), formater(ligne["montant non distribué (€)"]))


impactes = synthese[~synthese["statut"].isin(["Pas d'impact visible", "Hausse d'activité (report probable)"])]
afficher_chiffres({"GAB de la zone examinés": len(tri), "comportement changé (tri)": int(tri["statut"].isin(["Arrêté", "Baisse marquée", "Hausse marquée"]).sum()),
                   "GAB étudiés en détail": len(synthese), "impactés": len(impactes),
                   "hors service à ce jour": int((synthese["statut"] == "Hors service").sum()),
                   "jours d'indisponibilité cumulés": int(synthese["jours d'indisponibilité"].sum()),
                   "retraits non servis": int(impactes["retraits non servis"].sum()),
                   "montant non distribué (€)": int(impactes["montant non distribué (€)"].sum())}, "Impact des manifestations")
display(HTML(CSS + "<ul class='note' style='font-size:13px'>{}</ul>".format("".join("<li>{}</li>".format(phrase(l)) for _, l in synthese.iterrows()))))
display(HTML(CSS + "<div class='note'>Pour vérifier que la détection quotidienne les aurait signalés : copier ces GAB et leur date retenue dans "
             "<code>CAS_TESTS</code> du notebook <b>analyse_descriptive_fiche_gab</b> (section 7).</div>"))

In [ ]:
if DATASET_SORTIE_SYNTHESE:
    sortie = synthese.merge(tri[["GAB", "statut"]].rename(columns={"statut": "statut_tri"}), on="GAB", how="left")
    dataiku.Dataset(DATASET_SORTIE_SYNTHESE).write_with_schema(sortie)
    print("Synthèse écrite dans", DATASET_SORTIE_SYNTHESE)